# Etapa 5: Ingeniería de Características (Features) y Acople Demográfico

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 5: Feature Engineering) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, DAMA-BOK, PEP 8, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Acople Demográfico Oficial**: Integrar las series de población DANE 2018–2026 de [data/RAW/demografía/](file:///c:/Users/ADAN/OneDrive/Documentos/analisispapamercadoorlando/data/RAW/demograf%C3%ADa/).
2. **Cómputo de Indicadores Per Cápita**: Derivar el consumo aparente mayorista mensual en kg/hab.
3. **Índices de Estacionalidad**: Calcular el Índice Estacional de Precios (IEP) y de Oferta (IEO).
4. **Etiquetado de Anomalías No Destructivo**: Identificar outliers mediante criterios Tukey IQR y MAD Z-score modificado.
5. **Persistencia Medallion**: Exportar a `data/FEATURES/dataset_sipsa_features.parquet`.

In [ ]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    scipy>=1.10.0 \
    fastparquet>=2023.8.0 \
    tabulate>=0.9.0

print(f"[OK] Ambiente verificado en Python: {sys.version.split()[0]}")

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DE IMPORTS Y RUTAS DEL PROYECTO (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.infrastructure.config import CLEANED_DIR, FEATURES_DIR
from src.application.features_service import FeaturesService

print("Servicio de Features importado exitosamente.")

In [ ]:
# ==============================================================================
# CARGA DE DATOS SANEADOS DE LA CAPA CLEANED
# ==============================================================================
parquet_path = CLEANED_DIR / "dataset_sipsa_mensual_limpio.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    # Fallback con variabilidad temporal mensual
    fechas = pd.date_range("2022-01-01", "2024-12-01", freq="MS")
    rows = []
    for f in fechas:
        rows.append({
            "fecha_mes": f,
            "año": f.year,
            "mes": f.month,
            "divipola_depto": "25",
            "nombre_depto": "CUNDINAMARCA",
            "divipola_mpio": "25875",
            "nombre_mpio": "VILLAPINZON",
            "mercado_mayorista": "CORABASTOS",
            "alimento": "PAPA",
            "variedad_papa": "PAPA PASTUSA",
            "volumen_ingreso_ton": float(np.random.uniform(100, 300)),
            "precio_prom_kg": float(np.random.normal(2400, 300)),
            "precio_min_kg": 2000.0,
            "precio_max_kg": 2800.0,
            "num_transacciones": 28
        })
    df = pd.DataFrame(rows)

print(f"Filas listas para extracción de características: {len(df):,}")
display(df.head(4))

In [ ]:
# ==============================================================================
# EJECUCIÓN DEL PIPELINE DE FEATURES Y DEMOGRAFÍA
# ==============================================================================
feat_service = FeaturesService(df, output_dir=FEATURES_DIR)
df_features = feat_service.construir_panel_completo_features()

print("=== PANEL ENRIQUECIDO DE CARACTERÍSTICAS ===")
cols_muestra = [
    "fecha_mes", "año", "mes", "variedad_papa", "precio_prom_kg",
    "poblacion_nacional_cabecera", "consumo_mayorista_per_capita_kg",
    "indice_estacional_precio_iep", "es_outlier_iqr", "es_outlier_mad"
]
display(df_features[cols_muestra].head(6))

In [ ]:
# ==============================================================================
# RESUMEN DE INDICADORES PER CÁPITA Y ESTACIONALIDAD POR AÑO
# ==============================================================================
resumen_anual_feat = df_features.groupby("año").agg(
    consumo_mensual_promedio_kg=("consumo_mayorista_per_capita_kg", "mean"),
    iep_promedio=("indice_estacional_precio_iep", "mean"),
    total_outliers_iqr=("es_outlier_iqr", "sum"),
    total_outliers_mad=("es_outlier_mad", "sum")
).reset_index()

print("=== INDICADORES AGREGADOS ANUALES ===")
display(resumen_anual_feat)

### Conclusiones de la Etapa 5:
- Los denominadores poblacionales han normalizado la absorción del mercado por habitante.
- Los índices estacionales permiten aislar los ciclos agronómicos de cosecha (periodos de 120 vs 180 días).
- Las banderas de anomalías garantizan trazabilidad en la **Etapa 6 (Model)** sin necesidad de imputación destructiva.